# XAI stability

In [1]:
# Metrics for 
#        T-Explainer,
#        SHAP Explainer
#        KernelSHAP
#        SHAP ExactExplainer
#        LIME
#        Integrated Gradients
#        Input X Gradient
#        DeepLIFT
#        SmoothGrad (IG)
#        Vanilla Gradient
#        Guided Backprop
#        Occlusion

In [ ]:
import sys

!{sys.executable} -m pip install nbimporter
!{sys.executable} -m pip install tensorflow
!{sys.executable} -m pip install torch

#!git clone https://github.com/AI4LIFE-GROUP/OpenXAI.git
!{sys.executable} -m pip install -e OpenXAI

In [3]:
import time
import numpy as np
import pandas as pd
import nbimporter

import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import xgboost as xgb

import xai_explainers_defs as explainer

import Taylor_Explainer as texp


import warnings

In [4]:
import openxai

# Utils
import torch
import pickle

# Data loaders
from openxai.dataloader import return_loaders

# Perturbation methods required for the computation of the relative stability metrics
from openxai.explainers.catalog.perturbation_methods import NormalPerturbation
from openxai.explainers.catalog.perturbation_methods import NewDiscrete_NormalPerturbation

In [5]:
# Perturbation class parameters
perturbation_mean= 0.0
perturbation_std= 0.05
perturbation_flip_percentage= 0.01
    
perturbation= NormalPerturbation('tabular',
                                 mean=perturbation_mean,
                                 std_dev=perturbation_std,
                                 flip_percentage=perturbation_flip_percentage)

def generate_mask(explanation, top_k):
    mask_indices= torch.topk(explanation, top_k).indices
    mask= torch.zeros(explanation.shape) > 10
    for i in mask_indices:
        mask[i]= True
    return mask

# Quantitative Metrics -- Auxiliar Methods

In [6]:
# Create a custom PyTorch model that mimics the behavior of a scikit-learn MLPClassifier model

import torch.nn as nn

# Define the PyTorch Neural Network model
class MLPClassifierModel(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super(MLPClassifierModel, self).__init__()
        
        self.layers= nn.ModuleList([nn.Linear(input_size, hidden_sizes[0])])
        self.activations= [nn.ReLU()]
        
        for i in range(1, len(hidden_sizes)):
            self.layers.append(nn.Linear(hidden_sizes[i-1], hidden_sizes[i]))
            self.activations.append(nn.ReLU())
        
        self.output_layer= nn.Linear(hidden_sizes[-1], output_size)

        
    def forward(self, x):
        for layer, activation in zip(self.layers, self.activations):
            x= activation(layer(x))
        
        x= self.output_layer(x)
        
        return x

In [7]:
# convert a scikit-learn NN model to a PyTorch NN model

# skl_nn_model is the scikit-learn Neural Net model
# input_size is the number of input features

# RETURN a PyTorch Neural Net model used to binary classifications

def sklearn_to_pytorch_NN(skl_nn_model, input_size):

    # Convert the scikit-learn model to a PyTorch model
    input_size= input_size
    hidden_sizes= skl_nn_model.hidden_layer_sizes
    output_size= 1  # binary classification -- one output neuron for the binary prediction

    nn_pytorch_model= MLPClassifierModel(input_size, hidden_sizes, output_size)

    # Transfer the weights from the scikit-learn model to the PyTorch model
    for i, layer in enumerate(nn_pytorch_model.layers):
        layer.weight.data= torch.tensor(skl_nn_model.coefs_[i].T, dtype=torch.float32)
        layer.bias.data= torch.tensor(skl_nn_model.intercepts_[i], dtype=torch.float32)

    nn_pytorch_model.output_layer.weight.data= torch.tensor(skl_nn_model.coefs_[-1].T, dtype=torch.float32)
    nn_pytorch_model.output_layer.bias.data= torch.tensor(skl_nn_model.intercepts_[-1], dtype=torch.float32)
    
    
    return nn_pytorch_model

In [8]:
# bring explanations into data order (since LIME automatically orders according to highest importance)

def lime_exp_in_data_order(lime_exp, num_fts):
    
    exp= np.zeros(num_fts)

    for k, v in lime_exp.local_exp[1]:
        exp[k]= v

    return exp

In [9]:
# RETURN tensor_x and tensor_y datsets (tensors) according to euclidean distance ordering from target_x

def distance_ordering(tensor_x, tensor_y, target_x):
    
    # Calculate Euclidean distances for each row
    distances= torch.norm((tensor_x - target_x), dim=1)

    # Sort the data tensor based on distances
    sorted_indices= torch.argsort(distances)
    
    sorted_x= tensor_x[sorted_indices]
    sorted_y= tensor_y[sorted_indices]
    
    return sorted_x, sorted_y

In [10]:
# Remove a set of rows in a tensor dataset by index

# dataset is a n elements dataset
# index_to_remove is a m elements tensor with the indexes to remove

# RETURN a subset form dataset without the index_to_remove instances
    
def remove_tensor_row_by_indexset(dataset, index_to_remove):

    n_rows= index_to_remove.shape[0]
    
    index_to_remove= index_to_remove.sort().values
    
    subset= dataset.clone()
    
    for i in range(n_rows):
        row_exclude= index_to_remove[i]-i
    
        subset= torch.cat((subset[:row_exclude],subset[row_exclude+1:]))

    return subset

In [11]:
# Get a subset from a dataset with at least n_elements

# x is a tensor instance
# x_class is a tensor with the class of x
# dataset is a m elements tensor dataset
# dataset_class is a m elements tensor with the predicted 
# n_elements is an integer indicating the size of the subset

# RETURN two tensor subsets (from dataset and dataset_class) with 
#        option 1 - n_elements ordered first by class (same from x) and then by distance from x
#        option 0 - n_elements ordered by class (same from x) and filled (if necessary) with x and x_class

# option 0 gives us y' = y for all x' and option 1 relaxes such a restriction

def get_subsets(x, x_class, dataset, dataset_class, n_elements, option:int=0):
    data_size= dataset.shape[0]
    
    if (data_size< n_elements):
        raise ValueError("Data size must be greater than n_elements!")
    else:
        if (option):
            # order the dataset and dataset_class by distance from x
            dataset_order, dataset_class_order= distance_ordering(dataset, dataset_class, x.unsqueeze(0))
        
            # get the subset with first num_perts points by the same x class
            ind_same_class= (x_class == dataset_class_order).nonzero()[:n_elements].squeeze()
        
            subset= torch.index_select(input=dataset_order, dim=0, index=ind_same_class)
            subset_class= torch.index_select(input=dataset_class_order, dim=0, index=ind_same_class)
        else:
            # get the subset with first num_perts points by the same x class
            ind_same_class= (x_class == dataset_class).nonzero()[:n_elements].squeeze()
            
            # get only the elements in dataset under y' = y
            subset= torch.index_select(input=dataset, dim=0, index=ind_same_class)
            subset_class= torch.index_select(input=dataset_class, dim=0, index=ind_same_class)
            
            
        # if there are no elements enough in dataset matching with x_class
        if (subset.shape[0]< n_elements):
            last= n_elements - subset.shape[0]
            
            # we complete the n_elements of subset with the first instances of the ordered dataset
            if (option):
                if (ind_same_class.numel()== 1): # avoid a breaking when only one element matches with x_class
                    ind_same_class= torch.tensor([ind_same_class])

                dataset_order= remove_tensor_row_by_indexset(dataset_order, ind_same_class)
                dataset_class_order= remove_tensor_row_by_indexset(dataset_class_order, ind_same_class)

                dataset_order= dataset_order[0:last,:]
                dataset_class_order= dataset_class_order[0:last]

                subset= torch.cat((subset, dataset_order))
                subset_class= torch.cat((subset_class, dataset_class_order))
            
            # or we complete with x and x_class
            else:
                fill_x, fill_x_class = [], []
                
                for i in range(last):
                    fill_x.append(x)
                    fill_x_class.append(x_class)
                
                fill_x= torch.stack(fill_x)
                fill_x_class= torch.stack(fill_x_class)
                
                subset= torch.cat((subset, fill_x))
                subset_class= torch.cat((subset_class, fill_x_class.squeeze()))
            
            
        return subset, subset_class

In [70]:
ds_x= torch.tensor([[0.5885, 0.5961, 0.8704, 0.3738],
                    [0.5884, 0.5914, 0.8719, 0.3758],
                    [0.5905, 0.5933, 0.8656, 0.3686],
                    [0.5978, 0.5950, 0.8616, 0.3753],
                    [0.5881, 0.5873, 0.8663, 0.3726],
                    [0.5929, 0.5886, 0.8684, 0.3712],
                    [0.5914, 0.5882, 0.8652, 0.3822],
                    [0.5849, 0.6006, 0.8690, 0.3694],
                    [0.5849, 0.6032, 0.8677, 0.3707],
                    [0.5855, 0.5927, 0.8652, 0.3665],
                    [0.5841, 0.5866, 0.8631, 0.3747],
                    [0.5826, 0.6017, 0.8672, 0.3829],
                    [0.5848, 0.5873, 0.8670, 0.3827],
                    [0.5868, 0.6030, 0.8727, 0.3816],
                    [0.5804, 0.5916, 0.8600, 0.3738],
                    [0.5806, 0.6007, 0.8622, 0.3812],
                    [0.5885, 0.5875, 0.8711, 0.3688],
                    [0.5843, 0.5850, 0.8636, 0.3737],
                    [0.5833, 0.6043, 0.8595, 0.3764],
                    [0.5859, 0.6031, 0.8745, 0.3710],
                    [0.5995, 0.5965, 0.8580, 0.3724],
                    [0.5843, 0.5905, 0.8743, 0.3685],
                    [0.5829, 0.5999, 0.8645, 0.3653],
                    [0.5975, 0.5856, 0.8650, 0.3712],
                    [0.5912, 0.5915, 0.8624, 0.3636],
                    [0.5977, 0.6029, 0.8673, 0.3677],
                    [0.6002, 0.5916, 0.8576, 0.3744],
                    [0.5869, 0.5977, 0.8715, 0.3891],
                    [0.5946, 0.5898, 0.8552, 0.3822],
                    [0.6014, 0.5905, 0.8728, 0.3767]])

ds_y= torch.tensor([1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
                    0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
                    0, 0, 0, 0, 0, 0, 0, 1, 0, 0])

x_ponto= torch.tensor([0.5897, 0.5957, 0.8659, 0.3762])
x_ponto_class= torch.tensor([1])

get_subsets(x_ponto, x_ponto_class, ds_x, ds_y, 10)

(tensor([[0.5885, 0.5961, 0.8704, 0.3738],
         [0.5869, 0.5977, 0.8715, 0.3891],
         [0.5897, 0.5957, 0.8659, 0.3762],
         [0.5897, 0.5957, 0.8659, 0.3762],
         [0.5897, 0.5957, 0.8659, 0.3762],
         [0.5897, 0.5957, 0.8659, 0.3762],
         [0.5897, 0.5957, 0.8659, 0.3762],
         [0.5897, 0.5957, 0.8659, 0.3762],
         [0.5897, 0.5957, 0.8659, 0.3762],
         [0.5897, 0.5957, 0.8659, 0.3762]]),
 tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))

In [71]:
get_subsets(x_ponto, x_ponto_class, ds_x, ds_y, 10, 1)

(tensor([[0.5885, 0.5961, 0.8704, 0.3738],
         [0.5869, 0.5977, 0.8715, 0.3891],
         [0.5884, 0.5914, 0.8719, 0.3758],
         [0.5905, 0.5933, 0.8656, 0.3686],
         [0.5978, 0.5950, 0.8616, 0.3753],
         [0.5881, 0.5873, 0.8663, 0.3726],
         [0.5929, 0.5886, 0.8684, 0.3712],
         [0.5914, 0.5882, 0.8652, 0.3822],
         [0.5849, 0.6006, 0.8690, 0.3694],
         [0.5849, 0.6032, 0.8677, 0.3707]]),
 tensor([1, 1, 0, 0, 0, 0, 0, 0, 0, 0]))

In [12]:
# clip values near to zero in v replacing by eps

# v is a single value (float) or a numpy.ndarray with (n,) shape
# eps is a small number of tolerance limiting what is a small value

# RETURN v clipped

def clip_small_values(v, eps=1e-6):
    
    v_aux= v.copy()
    
    if (type(v_aux)== np.ndarray):
        elements= v_aux.shape[0]

        for i in range(elements):

            if (v_aux[i]< 0 and np.abs(v_aux[i])< eps):
                v_aux[i]= -eps
            elif (v_aux[i]> 0 and v_aux[i]< eps):
                v_aux[i]= eps
    else:
        if (v_aux< 0 and np.abs(v_aux)< eps):
            v_aux[i]= -eps
        elif (v_aux> 0 and v_aux< eps):
            v_aux= eps
                        
    return v_aux

In [13]:
# returns the square of the difference of any two quantities v1 and v2.

def square_difference(v1, v2):
    
    # arrays can be flattened, so long as ordering is preserved
    v1_flat= np.asarray(v1).flatten()
    v2_flat= np.asarray(v2).flatten()
    
    dif_flat= (v1_flat - v2_flat)
    
    return np.power(dif_flat, 2)

In [14]:
# returns the Lp norm of the difference between v1 and v2.
# normalizes the difference between v1 and v2 by v1 (adapted; Agarwal, Chirag, et al., 2022)

def lp_norm_dif(v1, v2, p_norm=2, eps=1e-6, norm:bool=True):
    
    # arrays can be flattened, so long as ordering is preserved
    v1_flat= np.asarray(v1).flatten()
    v2_flat= np.asarray(v2).flatten()
    
    dif_flat= (v1_flat - v2_flat)
    
    #if (norm==True): print('dif before div', dif_flat)
    
    if (norm==True):
        #v1_flat= np.clip(v1_flat, eps, None)
        v1_flat= clip_small_values(v1_flat, eps)
        
        dif_flat= np.divide(dif_flat, v1_flat, out=np.zeros_like(v1_flat), where=v1_flat!=0)
        
        #v2_flat= clip_small_values(v2_flat, eps)
        #dif_flat= 1 - np.divide(v2_flat, v1_flat, out=np.zeros_like(v1_flat), where=v1_flat!=0)
        
    #if (norm==True): print('dif afterr div', dif_flat)

    return np.linalg.norm(dif_flat, ord=p_norm)

In [15]:
# compute norm between predictions per perturbation - RIS

def ris_measure(x_data, x_pert, exp_data, exp_pert, p_norm=2, eps=1e-6):
    
    x_dif_norm= lp_norm_dif(x_data, x_pert, p_norm=p_norm, eps=eps, norm=True)
    #x_dif_norm= np.clip(x_dif_norm, eps, None)
    x_dif_norm= clip_small_values(x_dif_norm, eps)
    
    exp_dif_norm= lp_norm_dif(exp_data, exp_pert, p_norm=p_norm, eps=eps, norm=True)
    
    stability_measure= np.divide(exp_dif_norm, x_dif_norm, where=x_dif_norm!=0)
    
    """
    print('x_data', x_data)
    print('x_pert', x_pert)
    print('x_dif_norm', x_dif_norm)
    print('exp_ dif_norm', exp_dif_norm)
    print('stab_measure', stability_measure)
    """
    
    return stability_measure

In [16]:
# compute norm between representations - ROS

# x_data and x_pert must to be pd.DataFrame row individual instances with column names

def ros_measure(model, x_data, x_pert, exp_data, exp_pert, p_norm=2, eps=1e-6):
        
    fx_data= model.predict_proba(x_data)
    fx_pert= model.predict_proba(x_pert)
    
    fx_dif_norm= lp_norm_dif(fx_data, fx_pert, p_norm=p_norm, eps=eps, norm=True)
    #fx_dif_norm= np.clip(fx_dif_norm, eps, None)
    fx_dif_norm= clip_small_values(fx_dif_norm, eps)
    
    exp_dif_norm= lp_norm_dif(exp_data, exp_pert, p_norm=p_norm, eps=eps, norm=True)

    stability_measure= np.divide(exp_dif_norm, fx_dif_norm, where=fx_dif_norm!=0)
    
    """
    print('x_data', x_data)
    print('x_pert', x_pert)
    print('fx_data', fx_data)
    print('fx_pert', fx_pert)
    print('x_dif_norm', fx_dif_norm)
    print('exp_ dif_norm', exp_dif_norm)
    print('stab_measure', stability_measure)
    """
    
    return stability_measure

# Metric -- Relative Input/Output Stability -- RIS / ROS

In [17]:
import os
import pickle
from sklearn.base import clone

# Relative Input/Output Stability
# model is a treined classifier
# data is a preprocessed Pandas DataFrame -- model's training data
# labels are the data labels (Pandas DataFrame)
# perturbation is a OpenXAI perturbation object
# descriptor define the parameters to explanations and data perturbations
# cat_fts list indicating the categorical columns. if empty metric will consider all features as numeric
# model_is_NN True if model is a sklearn MLPClassifier, False if it is not 

# approximates the maximum L-p distance between explanations in a neighborhood around input x
# RETURN RIS max/mean and ROS max/mean metrics for 
#        T-Explainer,
#        SHAP Explainer
#        KernelSHAP
#        SHAP ExactExplainer
#        LIME
#        Integrated Gradients
#        Input X Gradient
#        DeepLIFT
#        SmoothGrad (IG)
#        Vanilla Gradient
#        Guided Backprop
#        Occlusion

def relative_stability(model, data, labels, perturbation, descriptor, cat_fts=[], is_model_NN:bool=False):
    
    tensor_train= torch.from_numpy(data.values)
    tensor_labels= torch.from_numpy(labels.values.ravel().astype(int))
    
    # ------------ data reduction for testing
    tensor_train = tensor_train[0:100, :]
    tensor_labels= tensor_labels[0:100]
    # ---------------------------------------
    
    t_ris_max_ratios= []
    shap_ris_max_ratios= []
    shap_krnel_ris_max_ratios= []
    shap_exact_ris_max_ratios= []
    lime_ris_max_ratios= []
    
    t_ris_mean_ratios= []
    shap_ris_mean_ratios= []
    shap_krnel_ris_mean_ratios= []
    shap_exact_ris_mean_ratios= []
    lime_ris_mean_ratios= []
    
    t_ros_max_ratios= []
    shap_ros_max_ratios= []
    shap_krnel_ros_max_ratios= []
    shap_exact_ros_max_ratios= []
    lime_ros_max_ratios= []
    
    t_ros_mean_ratios= []
    shap_ros_mean_ratios= []
    shap_krnel_ros_mean_ratios= []
    shap_exact_ros_mean_ratios= []
    lime_ros_mean_ratios= []
    
    if (is_model_NN==True):
        # convert a scikit-learn NN model to a PyTorch NN model used in captum
        nn_pytorch_model= sklearn_to_pytorch_NN(model, data.shape[1])
            
        itGd_ris_max_ratios= []
        iXGd_ris_max_ratios= []
        dLif_ris_max_ratios= []
        lwrp_ris_max_ratios= []
        smoo_ris_max_ratios= []
        vnGd_ris_max_ratios= []
        gdBp_ris_max_ratios= []
        occl_ris_max_ratios= []

        itGd_ris_mean_ratios= []
        iXGd_ris_mean_ratios= []
        dLif_ris_mean_ratios= []
        lwrp_ris_mean_ratios= []
        smoo_ris_mean_ratios= []
        vnGd_ris_mean_ratios= []
        gdBp_ris_mean_ratios= []
        occl_ris_mean_ratios= []

        itGd_ros_max_ratios= []
        iXGd_ros_max_ratios= []
        dLif_ros_max_ratios= []
        lwrp_ros_max_ratios= []
        smoo_ros_max_ratios= []
        vnGd_ros_max_ratios= []
        gdBp_ros_max_ratios= []
        occl_ros_max_ratios= []

        itGd_ros_mean_ratios= []
        iXGd_ros_mean_ratios= []
        dLif_ros_mean_ratios= []
        lwrp_ros_mean_ratios= []
        smoo_ros_mean_ratios= []
        vnGd_ros_mean_ratios= []
        gdBp_ros_mean_ratios= []
        occl_ros_mean_ratios= []
        
    
    for i_data, x_data in enumerate(tensor_train):
        # here x_data is tensor_train[i_data]
        
        # x_data and its label as pd.DataFrame
        target_i= pd.DataFrame(data=[x_data.numpy()], columns=data.columns)
        target_l= pd.DataFrame(data=[np.int64(tensor_labels[i_data])], columns=labels.columns)
        
        # data point prediction
        y_pred= torch.from_numpy(model.predict(target_i).astype(int))
        
        # ignore temporarily warnings related to feature names
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", message="X does not have valid feature names")
            
            # ------------------------------------ x_data explanation
            x_exp= explainer.Explainers()
            
            # data point explanation -- T-Exp
            t_x_exp= x_exp.t_exp(model, data, labels, target_i, target_l, descriptor, cat_fts=cat_fts)
            
            # data point explanation -- SHAP, TreeSHAP, KernelSHAP, and ExactSHAP
            if isinstance(model, xgb.XGBModel):
                shap_x_exp= x_exp.t_shap(model, target_i)
            else:
                shap_x_exp= x_exp.shap(model, data, target_i)
            
            shap_krnel_x_exp= x_exp.k_shap(model, data, target_i)
                
            if (data.shape[1]< 16):
                shap_exact_x_exp= x_exp.e_shap(model, data, target_i)

            # data point explanation -- LIME
            lime_x_exp= x_exp.lime(model, data, labels, target_i)
        
        # reset the warning settings
        warnings.resetwarnings()
        
        
        # data point explanation -- Gradient-based methods
        if (is_model_NN==True):
            x_data_tensor= torch.tensor(np.asarray(target_i), dtype=torch.float32)
            
            # ignore non-important warnings temporarily
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", message="Setting forward, backward hooks and attributes")
            
                itGd_x_exp= x_exp.int_grad(nn_pytorch_model, x_data_tensor)
                iXGd_x_exp= x_exp.inx_grad(nn_pytorch_model, x_data_tensor)
                dLif_x_exp= x_exp.dp_lift(nn_pytorch_model, x_data_tensor)
                lwrp_x_exp= x_exp.lrp(nn_pytorch_model, x_data_tensor)
                smoo_x_exp= x_exp.smo_grad(nn_pytorch_model, x_data_tensor, False) # Int. Grad.
                vnGd_x_exp= x_exp.vnl_grad(nn_pytorch_model, x_data_tensor)
                gdBp_x_exp= x_exp.g_bkprop(nn_pytorch_model, x_data_tensor)
                occl_x_exp= x_exp.occ(nn_pytorch_model, x_data_tensor)
            
            # reset the warning settings
            warnings.resetwarnings()
            

        # ------------------------------------ x_data perturbation
        # data point perturbation
        x_pert_samples= perturbation.get_perturbed_inputs(original_sample=x_data,
                                                          feature_mask=descriptor['mask'],
                                                          num_samples=descriptor['num_samples'],
                                                          max_distance=descriptor['pert_max_distance'],
                                                          feature_metadata=descriptor['feature_metadata'])

        # --- take the closest num_perts points to x_data that have the same predicted class label to x_data
        y_pert_preds= torch.from_numpy(model.predict(pd.DataFrame(data=x_pert_samples.numpy(),
                                                                 columns=data.columns)).astype(int))
        
        # get only the first num_perts points ordered by class and distance from x_data
        x_pert_samples, y_pert_preds= get_subsets(x_data, y_pred, x_pert_samples, y_pert_preds, 
                                                  descriptor['num_perts'])
        
        
        # ------------------------------------ explain each x_data perturbation
        t_exp_pert_samples= torch.zeros_like(x_pert_samples)
        shap_exp_pert_samples= torch.zeros_like(x_pert_samples)
        shap_krnel_exp_pert_samples= torch.zeros_like(x_pert_samples)
        shap_exact_exp_pert_samples= torch.zeros_like(x_pert_samples)
        lime_exp_pert_samples= torch.zeros_like(x_pert_samples)
        
        if (is_model_NN==True):
            itGd_exp_pert_samples= torch.zeros_like(x_pert_samples)
            iXGd_exp_pert_samples= torch.zeros_like(x_pert_samples)
            dLif_exp_pert_samples= torch.zeros_like(x_pert_samples)
            lwrp_exp_pert_samples= torch.zeros_like(x_pert_samples)
            smoo_exp_pert_samples= torch.zeros_like(x_pert_samples)
            vnGd_exp_pert_samples= torch.zeros_like(x_pert_samples)
            gdBp_exp_pert_samples= torch.zeros_like(x_pert_samples)
            occl_exp_pert_samples= torch.zeros_like(x_pert_samples)
        
        
        t_x_ris_ratios= []
        shap_x_ris_ratios= []
        shap_krnel_x_ris_ratios= []
        shap_exact_x_ris_ratios= []
        lime_x_ris_ratios= []
        
        t_x_ros_ratios= []
        shap_x_ros_ratios= []
        shap_krnel_x_ros_ratios= []
        shap_exact_x_ros_ratios= []
        lime_x_ros_ratios= []
        
        if (is_model_NN==True):
            itGd_x_ris_ratios= []
            iXGd_x_ris_ratios= []
            dLif_x_ris_ratios= []
            lwrp_x_ris_ratios= []
            smoo_x_ris_ratios= []
            vnGd_x_ris_ratios= []
            gdBp_x_ris_ratios= []
            occl_x_ris_ratios= []

            itGd_x_ros_ratios= []
            iXGd_x_ros_ratios= []
            dLif_x_ros_ratios= []
            lwrp_x_ros_ratios= []
            smoo_x_ros_ratios= []
            vnGd_x_ros_ratios= []
            gdBp_x_ros_ratios= []
            occl_x_ros_ratios= []
        
        """
        print('\nInstance', i_data)
        print('x', x_data)
        print('y_pred', y_pred)
        print('shap_exp', shap_x_exp)
        print('lime_exp', lime_x_exp)
        if (is_model_NN==True):
            print('itGd_x_exp', itGd_x_exp)
            print('iXGd_x_exp', iXGd_x_exp)
            print('dLif_x_exp', dLif_x_exp)
            print('lwrp_x_exp', lwrp_x_exp)
        print('x_pert_samples\n', x_pert_samples)
        print('y_pert_preds', y_pert_preds)
        #"""
        
        # For each perturbation, calculate the explanation
        for i, x_pert in enumerate(x_pert_samples):
            
            df_x_pert= pd.DataFrame(data=[x_pert.numpy()], columns=data.columns)
            df_y_pert= pd.DataFrame(data=[np.int64(y_pert_preds[i])], columns=labels.columns)
            
            # ignore temporarily warnings related to feature names
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", message="X does not have valid feature names")
            
                # ------------------------------------ x_pert explanation
                x_exp= explainer.Explainers()
                
                # perturbed data point explanation -- T-Exp
                t_exp_pert_samples[i, :]= x_exp.t_exp(model, data, labels, df_x_pert, df_y_pert, 
                                                      descriptor, cat_fts=cat_fts)
                
                # perturbed data point explanation -- SHAP, TreeSHAP, KernelSHAP, and ExactSHAP
                if isinstance(model, xgb.XGBModel):
                    shap_exp_pert_samples[i, :]= x_exp.t_shap(model, df_x_pert)
                else:
                    shap_exp_pert_samples[i, :]= x_exp.shap(model, data, df_x_pert)
                
                shap_krnel_exp_pert_samples[i, :]= x_exp.k_shap(model, data, df_x_pert)
                
                if (data.shape[1]< 16):
                    shap_exact_exp_pert_samples[i, :]= x_exp.e_shap(model, data, df_x_pert)

                # perturbed data point explanation -- LIME
                lime_exp_pert_samples[i, :]= x_exp.lime(model, data, labels, df_x_pert)
            
            # reset the warning settings
            warnings.resetwarnings()
            
            
            # perturbed data point explanation -- Gradient-based methods
            if (is_model_NN==True):
                x_pert_tensor= torch.tensor(np.asarray(df_x_pert), dtype=torch.float32)
                x_pert_tensor.requires_grad= True
                
                # ignore non-important warnings temporarily
                with warnings.catch_warnings():
                    warnings.filterwarnings("ignore", message="Setting forward, backward hooks and attributes")
                
                    itGd_exp_pert_samples[i, :]= x_exp.int_grad(nn_pytorch_model, x_pert_tensor)
                    iXGd_exp_pert_samples[i, :]= x_exp.inx_grad(nn_pytorch_model, x_pert_tensor)
                    dLif_exp_pert_samples[i, :]= x_exp.dp_lift(nn_pytorch_model, x_pert_tensor)
                    lwrp_exp_pert_samples[i, :]= x_exp.lrp(nn_pytorch_model, x_pert_tensor)
                    smoo_exp_pert_samples[i, :]= x_exp.smo_grad(nn_pytorch_model, x_pert_tensor, False) # Int. Grad.
                    vnGd_exp_pert_samples[i, :]= x_exp.vnl_grad(nn_pytorch_model, x_pert_tensor)
                    gdBp_exp_pert_samples[i, :]= x_exp.g_bkprop(nn_pytorch_model, x_pert_tensor)
                    occl_exp_pert_samples[i, :]= x_exp.occ(nn_pytorch_model, x_pert_tensor)
                
                # reset the warning settings
                warnings.resetwarnings()
        
    
            # ------------------------------------ get stability for each explanator and x_data perturbation           
            t_ris_measure= ris_measure(x_data, x_pert, 
                                       t_x_exp, t_exp_pert_samples[i], 
                                       p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            shap_ris_measure= ris_measure(x_data, x_pert, 
                                          shap_x_exp, shap_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            shap_krnel_ris_measure= ris_measure(x_data, x_pert, 
                                              shap_krnel_x_exp, shap_krnel_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            if (data.shape[1]< 16):
                shap_exact_ris_measure= ris_measure(x_data, x_pert, 
                                              shap_exact_x_exp, shap_exact_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            lime_ris_measure= ris_measure(x_data, x_pert, 
                                          lime_x_exp, lime_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            if (is_model_NN==True):
                itGd_ris_measure= ris_measure(x_data, x_pert, 
                                             itGd_x_exp, itGd_exp_pert_samples[i], 
                                             p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                iXGd_ris_measure= ris_measure(x_data, x_pert, 
                                              iXGd_x_exp, iXGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                dLif_ris_measure= ris_measure(x_data, x_pert, 
                                              dLif_x_exp, dLif_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                lwrp_ris_measure= ris_measure(x_data, x_pert, 
                                              lwrp_x_exp, lwrp_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                smoo_ris_measure= ris_measure(x_data, x_pert, 
                                              smoo_x_exp, smoo_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                vnGd_ris_measure= ris_measure(x_data, x_pert, 
                                              vnGd_x_exp, vnGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                gdBp_ris_measure= ris_measure(x_data, x_pert, 
                                              gdBp_x_exp, gdBp_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                occl_ris_measure= ris_measure(x_data, x_pert, 
                                              occl_x_exp, occl_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
            
            df_x_data= target_i.copy()
            
            t_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                          t_x_exp, t_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            shap_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                          shap_x_exp, shap_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            shap_krnel_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                          shap_krnel_x_exp, shap_krnel_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            if (data.shape[1]< 16):
                shap_exact_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                          shap_exact_x_exp, shap_exact_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            lime_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                          lime_x_exp, lime_exp_pert_samples[i], 
                                          p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
            
            if (is_model_NN==True):
                itGd_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                              itGd_x_exp, itGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                iXGd_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                              iXGd_x_exp, iXGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                dLif_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                              dLif_x_exp, dLif_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                lwrp_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                              lwrp_x_exp, lwrp_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                smoo_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                              smoo_x_exp, smoo_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                vnGd_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                              vnGd_x_exp, vnGd_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                gdBp_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                              gdBp_x_exp, gdBp_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
                
                occl_ros_measure= ros_measure(model, df_x_data, df_x_pert, 
                                              occl_x_exp, occl_exp_pert_samples[i], 
                                              p_norm=descriptor['p_norm'], eps=descriptor['eps_norm'])
        
            
            # --- stability measures for each x_data perturbation --- one processing cicle
            # RIS
            t_x_ris_ratios.append(t_ris_measure)
            shap_x_ris_ratios.append(shap_ris_measure)
            shap_krnel_x_ris_ratios.append(shap_krnel_ris_measure)
            lime_x_ris_ratios.append(lime_ris_measure)
            
            # ROS
            t_x_ros_ratios.append(t_ros_measure)
            shap_x_ros_ratios.append(shap_ros_measure)
            shap_krnel_x_ros_ratios.append(shap_krnel_ros_measure)
            lime_x_ros_ratios.append(lime_ros_measure)
            
            if (data.shape[1]< 16): 
                shap_exact_x_ris_ratios.append(shap_exact_ris_measure)
                shap_exact_x_ros_ratios.append(shap_exact_ros_measure)
                
            
            if (is_model_NN==True):
                itGd_x_ris_ratios.append(itGd_ris_measure)
                iXGd_x_ris_ratios.append(iXGd_ris_measure)
                dLif_x_ris_ratios.append(dLif_ris_measure)
                lwrp_x_ris_ratios.append(lwrp_ris_measure)
                smoo_x_ris_ratios.append(smoo_ris_measure)
                vnGd_x_ris_ratios.append(vnGd_ris_measure)
                gdBp_x_ris_ratios.append(gdBp_ris_measure)
                occl_x_ris_ratios.append(occl_ris_measure)

                itGd_x_ros_ratios.append(itGd_ros_measure)
                iXGd_x_ros_ratios.append(iXGd_ros_measure)
                dLif_x_ros_ratios.append(dLif_ros_measure)
                lwrp_x_ros_ratios.append(lwrp_ros_measure)
                smoo_x_ros_ratios.append(smoo_ros_measure)
                vnGd_x_ros_ratios.append(vnGd_ros_measure)
                gdBp_x_ros_ratios.append(gdBp_ros_measure)
                occl_x_ros_ratios.append(occl_ros_measure)
                
        """
        print('shap_exp_pert_samples\n', shap_exp_pert_samples)
        print('lime_exp_pert_samples\n', lime_exp_pert_samples)
        if (is_model_NN==True):
            print('itGd_exp_pert_samples\n', itGd_exp_pert_samples)
            print('iXGd_exp_pert_samples\n', iXGd_exp_pert_samples)
            print('dLif_exp_pert_samples\n', dLif_exp_pert_samples)
            print('lwrp_exp_pert_samples\n', lwrp_exp_pert_samples)
        #"""
        """  
        print('ris shap\n', shap_x_ris_ratios)
        print('ris lime\n', lime_x_ris_ratios)
        #"""
        
        # --- append only the max/mean value related to each x_data processed
        # max values
        t_ris_max_ratios.append(t_x_ris_ratios[np.argmax(t_x_ris_ratios)])
        shap_ris_max_ratios.append(shap_x_ris_ratios[np.argmax(shap_x_ris_ratios)])
        shap_krnel_ris_max_ratios.append(shap_krnel_x_ris_ratios[np.argmax(shap_krnel_x_ris_ratios)])
        lime_ris_max_ratios.append(lime_x_ris_ratios[np.argmax(lime_x_ris_ratios)])

        t_ros_max_ratios.append(t_x_ros_ratios[np.argmax(t_x_ros_ratios)])
        shap_ros_max_ratios.append(shap_x_ros_ratios[np.argmax(shap_x_ros_ratios)])
        shap_krnel_ros_max_ratios.append(shap_krnel_x_ros_ratios[np.argmax(shap_krnel_x_ros_ratios)])
        lime_ros_max_ratios.append(lime_x_ros_ratios[np.argmax(lime_x_ros_ratios)])
        
        # mean values
        t_ris_mean_ratios.append(np.mean(t_x_ris_ratios))
        shap_ris_mean_ratios.append(np.mean(shap_x_ris_ratios))
        shap_krnel_ris_mean_ratios.append(np.mean(shap_krnel_x_ris_ratios))
        lime_ris_mean_ratios.append(np.mean(lime_x_ris_ratios))

        t_ros_mean_ratios.append(np.mean(t_x_ros_ratios))
        shap_ros_mean_ratios.append(np.mean(shap_x_ros_ratios))
        shap_krnel_ros_mean_ratios.append(np.mean(shap_krnel_x_ros_ratios))
        lime_ros_mean_ratios.append(np.mean(lime_x_ros_ratios))
        
        if (data.shape[1]< 16): 
            # max values
            shap_exact_ris_max_ratios.append(shap_exact_x_ris_ratios[np.argmax(shap_exact_x_ris_ratios)])
            shap_exact_ros_max_ratios.append(shap_exact_x_ros_ratios[np.argmax(shap_exact_x_ros_ratios)])
            
            # mean values
            shap_exact_ris_mean_ratios.append(np.mean(shap_exact_x_ris_ratios))
            shap_exact_ros_mean_ratios.append(np.mean(shap_exact_x_ros_ratios))
            
        
        if (is_model_NN==True):
            # max values
            itGd_ris_max_ratios.append(itGd_x_ris_ratios[np.argmax(itGd_x_ris_ratios)])
            iXGd_ris_max_ratios.append(iXGd_x_ris_ratios[np.argmax(iXGd_x_ris_ratios)])
            dLif_ris_max_ratios.append(dLif_x_ris_ratios[np.argmax(dLif_x_ris_ratios)])
            lwrp_ris_max_ratios.append(lwrp_x_ris_ratios[np.argmax(lwrp_x_ris_ratios)])
            smoo_ris_max_ratios.append(smoo_x_ris_ratios[np.argmax(smoo_x_ris_ratios)])
            vnGd_ris_max_ratios.append(vnGd_x_ris_ratios[np.argmax(vnGd_x_ris_ratios)])
            gdBp_ris_max_ratios.append(gdBp_x_ris_ratios[np.argmax(gdBp_x_ris_ratios)])
            occl_ris_max_ratios.append(occl_x_ris_ratios[np.argmax(occl_x_ris_ratios)])

            itGd_ros_max_ratios.append(itGd_x_ros_ratios[np.argmax(itGd_x_ros_ratios)])
            iXGd_ros_max_ratios.append(iXGd_x_ros_ratios[np.argmax(iXGd_x_ros_ratios)])
            dLif_ros_max_ratios.append(dLif_x_ros_ratios[np.argmax(dLif_x_ros_ratios)])
            lwrp_ros_max_ratios.append(lwrp_x_ros_ratios[np.argmax(lwrp_x_ros_ratios)])
            smoo_ros_max_ratios.append(smoo_x_ros_ratios[np.argmax(smoo_x_ros_ratios)])
            vnGd_ros_max_ratios.append(vnGd_x_ros_ratios[np.argmax(vnGd_x_ros_ratios)])
            gdBp_ros_max_ratios.append(gdBp_x_ros_ratios[np.argmax(gdBp_x_ros_ratios)])
            occl_ros_max_ratios.append(occl_x_ros_ratios[np.argmax(occl_x_ros_ratios)])

            # mean values
            itGd_ris_mean_ratios.append(np.mean(itGd_x_ris_ratios))
            iXGd_ris_mean_ratios.append(np.mean(iXGd_x_ris_ratios))
            dLif_ris_mean_ratios.append(np.mean(dLif_x_ris_ratios))
            lwrp_ris_mean_ratios.append(np.mean(lwrp_x_ris_ratios))
            smoo_ris_mean_ratios.append(np.mean(smoo_x_ris_ratios))
            vnGd_ris_mean_ratios.append(np.mean(vnGd_x_ris_ratios))
            gdBp_ris_mean_ratios.append(np.mean(gdBp_x_ris_ratios))
            occl_ris_mean_ratios.append(np.mean(occl_x_ris_ratios))

            itGd_ros_mean_ratios.append(np.mean(itGd_x_ros_ratios))
            iXGd_ros_mean_ratios.append(np.mean(iXGd_x_ros_ratios))
            dLif_ros_mean_ratios.append(np.mean(dLif_x_ros_ratios))
            lwrp_ros_mean_ratios.append(np.mean(lwrp_x_ros_ratios))
            smoo_ros_mean_ratios.append(np.mean(smoo_x_ros_ratios))
            vnGd_ros_mean_ratios.append(np.mean(vnGd_x_ros_ratios))
            gdBp_ros_mean_ratios.append(np.mean(gdBp_x_ros_ratios))
            occl_ros_mean_ratios.append(np.mean(occl_x_ros_ratios))
           
    # ------------------------------------ RETURN ratios considering all data processed
    t_ris_max = t_ris_max_ratios[np.argmax(t_ris_max_ratios)]
    shap_ris_max= shap_ris_max_ratios[np.argmax(shap_ris_max_ratios)]
    shap_krnel_ris_max= shap_krnel_ris_max_ratios[np.argmax(shap_krnel_ris_max_ratios)]
    lime_ris_max= lime_ris_max_ratios[np.argmax(lime_ris_max_ratios)]
    
    t_ris_max_std = np.std(t_ris_max_ratios)
    shap_ris_max_std= np.std(shap_ris_max_ratios)
    shap_krnel_ris_max_std= np.std(shap_krnel_ris_max_ratios)
    lime_ris_max_std= np.std(lime_ris_max_ratios)
    
    t_ros_max = t_ros_max_ratios[np.argmax(t_ros_max_ratios)]
    shap_ros_max= shap_ros_max_ratios[np.argmax(shap_ros_max_ratios)]
    shap_krnel_ros_max= shap_krnel_ros_max_ratios[np.argmax(shap_krnel_ros_max_ratios)]
    lime_ros_max= lime_ros_max_ratios[np.argmax(lime_ros_max_ratios)]
    
    t_ros_max_std = np.std(t_ros_max_ratios)
    shap_ros_max_std= np.std(shap_ros_max_ratios)
    shap_krnel_ros_max_std= np.std(shap_krnel_ros_max_ratios)
    lime_ros_max_std= np.std(lime_ros_max_ratios)
    
    
    t_ris_mean = np.mean(t_ris_mean_ratios)
    shap_ris_mean= np.mean(shap_ris_mean_ratios)
    shap_krnel_ris_mean= np.mean(shap_krnel_ris_mean_ratios)
    lime_ris_mean= np.mean(lime_ris_mean_ratios)
    
    t_ris_mean_std = np.std(t_ris_mean_ratios)
    shap_ris_mean_std= np.std(shap_ris_mean_ratios)
    shap_krnel_ris_mean_std= np.std(shap_krnel_ris_mean_ratios)
    lime_ris_mean_std= np.std(lime_ris_mean_ratios)
    
    t_ros_mean = np.mean(t_ros_mean_ratios)
    shap_ros_mean= np.mean(shap_ros_mean_ratios)
    shap_krnel_ros_mean= np.mean(shap_krnel_ros_mean_ratios)
    lime_ros_mean= np.mean(lime_ros_mean_ratios)
    
    t_ros_mean_std = np.std(t_ros_mean_ratios)
    shap_ros_mean_std= np.std(shap_ros_mean_ratios)
    shap_krnel_ros_mean_std= np.std(shap_krnel_ros_mean_ratios)
    lime_ros_mean_std= np.std(lime_ros_mean_ratios)
    
    if (data.shape[1]< 16):
        shap_exact_ris_max= shap_exact_ris_max_ratios[np.argmax(shap_exact_ris_max_ratios)]
        shap_exact_ris_max_std= np.std(shap_exact_ris_max_ratios)

        shap_exact_ros_max= shap_exact_ros_max_ratios[np.argmax(shap_exact_ros_max_ratios)]
        shap_exact_ros_max_std= np.std(shap_exact_ros_max_ratios)

        shap_exact_ris_mean= np.mean(shap_exact_ris_mean_ratios)
        shap_exact_ris_mean_std= np.std(shap_exact_ris_mean_ratios)

        shap_exact_ros_mean= np.mean(shap_exact_ros_mean_ratios)
        shap_exact_ros_mean_std= np.std(shap_exact_ros_mean_ratios)
        
    
    if (is_model_NN==True):
        itGd_ris_max= itGd_ris_max_ratios[np.argmax(itGd_ris_max_ratios)]
        iXGd_ris_max= iXGd_ris_max_ratios[np.argmax(iXGd_ris_max_ratios)]
        dLif_ris_max= dLif_ris_max_ratios[np.argmax(dLif_ris_max_ratios)]
        lwrp_ris_max= lwrp_ris_max_ratios[np.argmax(lwrp_ris_max_ratios)]
        smoo_ris_max= smoo_ris_max_ratios[np.argmax(smoo_ris_max_ratios)]
        vnGd_ris_max= vnGd_ris_max_ratios[np.argmax(vnGd_ris_max_ratios)]
        gdBp_ris_max= gdBp_ris_max_ratios[np.argmax(gdBp_ris_max_ratios)]
        occl_ris_max= occl_ris_max_ratios[np.argmax(occl_ris_max_ratios)]

        itGd_ris_max_std= np.std(itGd_ris_max_ratios)
        iXGd_ris_max_std= np.std(iXGd_ris_max_ratios)
        dLif_ris_max_std= np.std(dLif_ris_max_ratios)
        lwrp_ris_max_std= np.std(lwrp_ris_max_ratios)
        smoo_ris_max_std= np.std(smoo_ris_max_ratios)
        vnGd_ris_max_std= np.std(vnGd_ris_max_ratios)
        gdBp_ris_max_std= np.std(gdBp_ris_max_ratios)
        occl_ris_max_std= np.std(occl_ris_max_ratios)

        itGd_ros_max= itGd_ros_max_ratios[np.argmax(itGd_ros_max_ratios)]
        iXGd_ros_max= iXGd_ros_max_ratios[np.argmax(iXGd_ros_max_ratios)]
        dLif_ros_max= dLif_ros_max_ratios[np.argmax(dLif_ros_max_ratios)]
        lwrp_ros_max= lwrp_ros_max_ratios[np.argmax(lwrp_ros_max_ratios)]
        smoo_ros_max= smoo_ros_max_ratios[np.argmax(smoo_ros_max_ratios)]
        vnGd_ros_max= vnGd_ros_max_ratios[np.argmax(vnGd_ros_max_ratios)]
        gdBp_ros_max= gdBp_ros_max_ratios[np.argmax(gdBp_ros_max_ratios)]
        occl_ros_max= occl_ros_max_ratios[np.argmax(occl_ros_max_ratios)]

        itGd_ros_max_std= np.std(itGd_ros_max_ratios)
        iXGd_ros_max_std= np.std(iXGd_ros_max_ratios)
        dLif_ros_max_std= np.std(dLif_ros_max_ratios)
        lwrp_ros_max_std= np.std(lwrp_ros_max_ratios)
        smoo_ros_max_std= np.std(smoo_ros_max_ratios)
        vnGd_ros_max_std= np.std(vnGd_ros_max_ratios)
        gdBp_ros_max_std= np.std(gdBp_ros_max_ratios)
        occl_ros_max_std= np.std(occl_ros_max_ratios)
        
        
        itGd_ris_mean= np.mean(itGd_ris_mean_ratios)
        iXGd_ris_mean= np.mean(iXGd_ris_mean_ratios)
        dLif_ris_mean= np.mean(dLif_ris_mean_ratios)
        lwrp_ris_mean= np.mean(lwrp_ris_mean_ratios)
        smoo_ris_mean= np.mean(smoo_ris_mean_ratios)
        vnGd_ris_mean= np.mean(vnGd_ris_mean_ratios)
        gdBp_ris_mean= np.mean(gdBp_ris_mean_ratios)
        occl_ris_mean= np.mean(occl_ris_mean_ratios)

        itGd_ris_mean_std= np.std(itGd_ris_mean_ratios)
        iXGd_ris_mean_std= np.std(iXGd_ris_mean_ratios)
        dLif_ris_mean_std= np.std(dLif_ris_mean_ratios)
        lwrp_ris_mean_std= np.std(lwrp_ris_mean_ratios)
        smoo_ris_mean_std= np.std(smoo_ris_mean_ratios)
        vnGd_ris_mean_std= np.std(vnGd_ris_mean_ratios)
        gdBp_ris_mean_std= np.std(gdBp_ris_mean_ratios)
        occl_ris_mean_std= np.std(occl_ris_mean_ratios)

        itGd_ros_mean= np.mean(itGd_ros_mean_ratios)
        iXGd_ros_mean= np.mean(iXGd_ros_mean_ratios)
        dLif_ros_mean= np.mean(dLif_ros_mean_ratios)
        lwrp_ros_mean= np.mean(lwrp_ros_mean_ratios)
        smoo_ros_mean= np.mean(smoo_ros_mean_ratios)
        vnGd_ros_mean= np.mean(vnGd_ros_mean_ratios)
        gdBp_ros_mean= np.mean(gdBp_ros_mean_ratios)
        occl_ros_mean= np.mean(occl_ros_mean_ratios)

        itGd_ros_mean_std= np.std(itGd_ros_mean_ratios)
        iXGd_ros_mean_std= np.std(iXGd_ros_mean_ratios)
        dLif_ros_mean_std= np.std(dLif_ros_mean_ratios)
        lwrp_ros_mean_std= np.std(lwrp_ros_mean_ratios)
        smoo_ros_mean_std= np.std(smoo_ros_mean_ratios)
        vnGd_ros_mean_std= np.std(vnGd_ros_mean_ratios)
        gdBp_ros_mean_std= np.std(gdBp_ros_mean_ratios)
        occl_ros_mean_std= np.std(occl_ros_mean_ratios)
        
        
    results= {
        't_exp_ris_max': t_ris_max, 'std(t_exp_ris_max)': t_ris_max_std,
        'shap_ris_max': shap_ris_max, 'std(shap_ris_max)': shap_ris_max_std,
        'lime_ris_max': lime_ris_max, 'std(lime_ris_max)': lime_ris_max_std,
        't_exp_ris_mean': t_ris_mean, 'std(t_exp_ris_mean)': t_ris_mean_std,
        'shap_ris_mean': shap_ris_mean, 'std(shap_ris_mean)': shap_ris_mean_std,
        'lime_ris_mean': lime_ris_mean, 'std(lime_ris_mean)': lime_ris_mean_std,
        't_exp_ros_max': t_ros_max, 'std(t_exp_ros_max)': t_ros_max_std,
        'shap_ros_max': shap_ros_max, 'std(shap_ros_max)': shap_ros_max_std,
        'lime_ros_max': lime_ros_max, 'std(lime_ros_max)': lime_ros_max_std,
        't_exp_ros_mean': t_ros_mean, 'std(t_exp_ros_mean)': t_ros_mean_std,
        'shap_ros_mean': shap_ros_mean, 'std(shap_ros_mean)': shap_ros_mean_std,
        'lime_ros_mean': lime_ros_mean, 'std(lime_ros_mean)': lime_ros_mean_std,
        'shap_kernel_ris_max': shap_krnel_ris_max, 'std(shap_kernel_ris_max)': shap_krnel_ris_max_std,
        'shap_kernel_ris_mean': shap_krnel_ris_mean, 'std(shap_kernel_ris_mean)': shap_krnel_ris_mean_std,
        'shap_kernel_ros_max': shap_krnel_ros_max, 'std(shap_kernel_ros_max)': shap_krnel_ros_max_std,
        'shap_kernel_ros_mean': shap_krnel_ros_mean, 'std(shap_kernel_ros_mean)': shap_krnel_ros_mean_std,
    }
    
    if (data.shape[1]< 16):
        results_sh_exact= {
            'shap_exact_ris_max': shap_exact_ris_max, 'std(shap_exact_ris_max)': shap_exact_ris_max_std,
            'shap_exact_ris_mean': shap_exact_ris_mean, 'std(shap_exact_ris_mean)': shap_exact_ris_mean_std,
            'shap_exact_ros_max': shap_exact_ros_max, 'std(shap_exact_ros_max)': shap_exact_ros_max_std,
            'shap_exact_ros_mean': shap_exact_ros_mean, 'std(shap_exact_ros_mean)': shap_exact_ros_mean_std,
        }
        
        results.update(results_sh_exact)
        

    if (is_model_NN==True):
        results_grad= {
            'itGd_ris_max': itGd_ris_max, 'std(itGd_ris_max)': itGd_ris_max_std,
            'iXGd_ris_max': iXGd_ris_max, 'std(iXGd_ris_max)': iXGd_ris_max_std,
            'dLif_ris_max': dLif_ris_max, 'std(dLif_ris_max)': dLif_ris_max_std,
            'lwrp_ris_max': lwrp_ris_max, 'std(lwrp_ris_max)': lwrp_ris_max_std,
            'smoothG_ris_max': smoo_ris_max, 'std(smoothG_ris_max)': smoo_ris_max_std,
            'vanillaG_ris_max': vnGd_ris_max, 'std(vanillaG_ris_max)': vnGd_ris_max_std,
            'GuidBprop_ris_max': gdBp_ris_max, 'std(GuidBprop_ris_max)': gdBp_ris_max_std,
            'occlusion_ris_max': occl_ris_max, 'std(occlusion_ris_max)': occl_ris_max_std,
            
            'itGd_ris_mean': itGd_ris_mean, 'std(itGd_ris_mean)': itGd_ris_mean_std,
            'iXGd_ris_mean': iXGd_ris_mean, 'std(iXGd_ris_mean)': iXGd_ris_mean_std,
            'dLif_ris_mean': dLif_ris_mean, 'std(dLif_ris_mean)': dLif_ris_mean_std,
            'lwrp_ris_mean': lwrp_ris_mean, 'std(lwrp_ris_mean)': lwrp_ris_mean_std,
            'smoothG_ris_mean': smoo_ris_mean, 'std(smoothG_ris_mean)': smoo_ris_mean_std,
            'vanillaG_ris_mean': vnGd_ris_mean, 'std(vanillaG_ris_mean)': vnGd_ris_mean_std,
            'GuidBprop_ris_mean': gdBp_ris_mean, 'std(GuidBprop_ris_mean)': gdBp_ris_mean_std,
            'occlusion_ris_mean': occl_ris_mean, 'std(occlusion_ris_mean)': occl_ris_mean_std,
            
            'itGd_ros_max': itGd_ros_max, 'std(itGd_ros_max)': itGd_ros_max_std,
            'iXGd_ros_max': iXGd_ros_max, 'std(iXGd_ros_max)': iXGd_ros_max_std,
            'dLif_ros_max': dLif_ros_max, 'std(dLif_ros_max)': dLif_ros_max_std,
            'lwrp_ros_max': lwrp_ros_max, 'std(lwrp_ros_max)': lwrp_ros_max_std,
            'smoothG_ros_max': smoo_ros_max, 'std(smoothG_ros_max)': smoo_ros_max_std,
            'vanillaG_ros_max': vnGd_ros_max, 'std(vanillaG_ros_max)': vnGd_ros_max_std,
            'GuidBprop_ros_max': gdBp_ros_max, 'std(GuidBprop_ros_max)': gdBp_ros_max_std,
            'occlusion_ros_max': occl_ros_max, 'std(occlusion_ros_max)': occl_ros_max_std,
            
            'itGd_ros_mean': itGd_ros_mean, 'std(itGd_ros_mean)': itGd_ros_mean_std,
            'iXGd_ros_mean': iXGd_ros_mean, 'std(iXGd_ros_mean)': iXGd_ros_mean_std,
            'dLif_ros_mean': dLif_ros_mean, 'std(dLif_ros_mean)': dLif_ros_mean_std,
            'lwrp_ros_mean': lwrp_ros_mean, 'std(lwrp_ros_mean)': lwrp_ros_mean_std,
            'smoothG_ros_mean': smoo_ros_mean, 'std(smoothG_ros_mean)': smoo_ros_mean_std,
            'vanillaG_ros_mean': vnGd_ros_mean, 'std(vanillaG_ros_mean)': vnGd_ros_mean_std,
            'GuidBprop_ros_mean': gdBp_ros_mean, 'std(GuidBprop_ros_mean)': gdBp_ros_mean_std,
            'occlusion_ros_mean': occl_ros_mean, 'std(occlusion_ros_mean)': occl_ros_mean_std
        }

        results.update(results_grad)

    # the max/mean stability ratios
    return results

In [43]:
# evaluate relative input/output stability -- using the 2 first instances from train_ox
print('RIS/ROS --') 
relative_stability(nn3_model_ox, train_ox, labels_train_ox, perturbation, descriptor_ox, is_model_NN=True)

RIS/ROS --


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


{'t_exp_ris_max': 24.589273992307902,
 'std(t_exp_ris_max)': 5.624717017028873,
 'shap_ris_max': 60.14265195213151,
 'std(shap_ris_max)': 23.172793328396292,
 'lime_ris_max': 4.857318869298499,
 'std(lime_ris_max)': 0.691981500344578,
 't_exp_ris_mean': 8.665496239748043,
 'std(t_exp_ris_mean)': 3.0135497606511352,
 'shap_ris_mean': 19.262508283107785,
 'std(shap_ris_mean)': 11.049357612400339,
 'lime_ris_mean': 2.0208849835573397,
 'std(lime_ris_mean)': 0.4046528923104392,
 't_exp_ros_max': 36983.59130661488,
 'std(t_exp_ros_max)': 18181.7733299686,
 'shap_ros_max': 79979.60494254815,
 'std(shap_ros_max)': 39289.164593036374,
 'lime_ros_max': 3161.6874142196107,
 'std(lime_ros_max)': 1536.1142397927263,
 't_exp_ros_mean': 975.9537780977139,
 'std(t_exp_ros_mean)': 921.235782545536,
 'shap_ros_mean': 2103.1133111617846,
 'std(shap_ros_mean)': 1991.737859099798,
 'lime_ros_mean': 91.81147203957521,
 'std(lime_ros_mean)': 82.01887542248149,
 'shap_kernel_ris_max': 19.430072446685813,
 's

# Metric -- Run Explanation Stability -- RES

In [18]:
# Run Explanation Stability
# model is a treined classifier
# data is a preprocessed Pandas DataFrame -- model's training data
# labels are the data labels (Pandas DataFrame)
# descriptor define the parameters to explanations and data perturbations
# cat_fts list indicating the categorical columns. if empty metric will consider all features as numeric

# RETURN a measure of stability from multiple runs over non-perturbed x.
# the greater the value, the less stable the method is. Methods:
#        T-Explainer,
#        SHAP Explainer
#        KernelSHAP
#        SHAP ExactExplainer
#        LIME
#        Integrated Gradients
#        Input X Gradient
#        DeepLIFT
#        SmoothGrad (IG)
#        Vanilla Gradient
#        Guided Backprop
#        Occlusion

def run_stability(model, data, labels, descriptor, cat_fts=[], is_model_NN:bool=False):
    
    tensor_train= torch.from_numpy(data.values)
    tensor_labels= torch.from_numpy(labels.values.ravel().astype(int))
    
    # ------------ data reduction for testing
    tensor_train = tensor_train[0:100, :]
    tensor_labels= tensor_labels[0:100]
    # ---------------------------------------
    
    t_stability_ratios= []
    shap_stability_ratios= []
    shap_krnel_stability_ratios= []
    shap_exact_stability_ratios= []
    lime_stability_ratios= []
    
    if (is_model_NN==True):
        # convert a scikit-learn NN model to a PyTorch NN model used in captum
        nn_pytorch_model= sklearn_to_pytorch_NN(model, data.shape[1])
            
        itGd_stability_ratios= []
        iXGd_stability_ratios= []
        dLif_stability_ratios= []
        lwrp_stability_ratios= []
        smoo_stability_ratios= []
        vnGd_stability_ratios= []
        gdBp_stability_ratios= []
        occl_stability_ratios= []
        
    
    runs= int(descriptor['num_runs'])
    
    for i_data, x_data in enumerate(tensor_train):
        # here x_data is tensor_train[i_data]

        # x_data and its label as pd.DataFrame
        target_i= pd.DataFrame(x_data, data.columns).T
        target_l= pd.DataFrame(data=[np.int64(tensor_labels[i_data])], columns=labels.columns)
        
        y_pred= model.predict(target_i).astype(int)[0]

        t_x_exps= []
        shap_x_exps= []
        shap_krnel_x_exps= []
        shap_exact_x_exps= []
        lime_x_exps= []
        
        if (is_model_NN==True):
            itGd_x_exps= []
            iXGd_x_exps= []
            dLif_x_exps= []
            lwrp_x_exps= []
            smoo_x_exps= []
            vnGd_x_exps= []
            gdBp_x_exps= []
            occl_x_exps= []
                               
        for i in range(runs):
            # ignore temporarily warnings related to feature names
            with warnings.catch_warnings():
                warnings.filterwarnings("ignore", message="X does not have valid feature names")
            
                # ------------------------------------ n runs x_data explanation
                x_exp= explainer.Explainers()
            
                # data point explanation -- T-Exp
                t_x_exp= x_exp.t_exp(model, data, labels, target_i, target_l, descriptor, cat_fts=cat_fts)

                # data point explanation -- SHAP, TreeSHAP, KernelSHAP, and ExactSHAP
                if isinstance(model, xgb.XGBModel):
                    shap_x_exp= x_exp.t_shap(model, target_i)
                else:
                    shap_x_exp= x_exp.shap(model, data, target_i)

                shap_krnel_x_exp= x_exp.k_shap(model, data, target_i)

                if (data.shape[1]< 16):
                    shap_exact_x_exp= x_exp.e_shap(model, data, target_i)

                # data point explanation -- LIME
                lime_x_exp= x_exp.lime(model, data, labels, target_i)

            # reset the warning settings
            warnings.resetwarnings()
            
            
            # data point explanation -- Gradient-based methods
            if (is_model_NN==True):
                x_data_tensor= torch.tensor(np.asarray(target_i), dtype=torch.float32)
                x_data_tensor.requires_grad= True

                # ignore non-important warnings temporarily
                with warnings.catch_warnings():
                    warnings.filterwarnings("ignore", message="Setting forward, backward hooks and attributes")

                    itGd_x_exp= x_exp.int_grad(nn_pytorch_model, x_data_tensor)
                    iXGd_x_exp= x_exp.inx_grad(nn_pytorch_model, x_data_tensor)
                    dLif_x_exp= x_exp.dp_lift(nn_pytorch_model, x_data_tensor)
                    lwrp_x_exp= x_exp.lrp(nn_pytorch_model, x_data_tensor)
                    smoo_x_exp= x_exp.smo_grad(nn_pytorch_model, x_data_tensor, False) # Int. Grad.
                    vnGd_x_exp= x_exp.vnl_grad(nn_pytorch_model, x_data_tensor)
                    gdBp_x_exp= x_exp.g_bkprop(nn_pytorch_model, x_data_tensor)
                    occl_x_exp= x_exp.occ(nn_pytorch_model, x_data_tensor)

                # reset the warning settings
                warnings.resetwarnings()
            
            # get the explanation of each method to each run
            t_x_exps.append(t_x_exp.numpy())
            shap_x_exps.append(shap_x_exp.numpy())
            shap_krnel_x_exps.append(shap_krnel_x_exp.numpy())
            lime_x_exps.append(lime_x_exp.numpy())
            
            if (data.shape[1]< 16):
                shap_exact_x_exps.append(shap_exact_x_exp.numpy())
            
            if (is_model_NN==True):
                itGd_x_exps.append(itGd_x_exp.numpy())
                iXGd_x_exps.append(iXGd_x_exp.numpy())
                dLif_x_exps.append(dLif_x_exp.numpy())
                lwrp_x_exps.append(lwrp_x_exp.numpy())
                smoo_x_exps.append(smoo_x_exp.numpy())
                vnGd_x_exps.append(vnGd_x_exp.numpy())
                gdBp_x_exps.append(gdBp_x_exp.numpy())
                occl_x_exps.append(occl_x_exp.numpy())
                
        
        t_x_exps= np.asarray(t_x_exps)
        shap_x_exps= np.asarray(shap_x_exps)
        shap_krnel_x_exps= np.asarray(shap_krnel_x_exps)
        lime_x_exps= np.asarray(lime_x_exps)
        
        t_x_exps_mean= np.mean(t_x_exps, axis=0)
        shap_x_exps_mean= np.mean(shap_x_exps, axis=0)
        shap_krnel_x_exps_mean= np.mean(shap_krnel_x_exps, axis=0)
        lime_x_exps_mean= np.mean(lime_x_exps, axis=0)
        
        t_x_exp_ratios= []
        shap_x_exp_ratios= []
        shap_krnel_x_exp_ratios= []
        lime_x_exp_ratios= []
        
        if (data.shape[1]< 16):
            shap_exact_x_exps= np.asarray(shap_exact_x_exps)
            shap_exact_x_exps_mean= np.mean(shap_exact_x_exps, axis=0)
            
            shap_exact_x_exp_ratios= []
        
        
        if (is_model_NN==True):
            itGd_x_exps= np.asarray(itGd_x_exps)
            iXGd_x_exps= np.asarray(iXGd_x_exps)
            dLif_x_exps= np.asarray(dLif_x_exps)
            lwrp_x_exps= np.asarray(lwrp_x_exps)
            smoo_x_exps= np.asarray(smoo_x_exps)
            vnGd_x_exps= np.asarray(vnGd_x_exps)
            gdBp_x_exps= np.asarray(gdBp_x_exps)
            occl_x_exps= np.asarray(occl_x_exps)
            
            itGd_x_exps_mean= np.mean(itGd_x_exps, axis=0)
            iXGd_x_exps_mean= np.mean(iXGd_x_exps, axis=0)
            dLif_x_exps_mean= np.mean(dLif_x_exps, axis=0)
            lwrp_x_exps_mean= np.mean(lwrp_x_exps, axis=0)
            smoo_x_exps_mean= np.mean(smoo_x_exps, axis=0)
            vnGd_x_exps_mean= np.mean(vnGd_x_exps, axis=0)
            gdBp_x_exps_mean= np.mean(gdBp_x_exps, axis=0)
            occl_x_exps_mean= np.mean(occl_x_exps, axis=0)
            
            itGd_x_exp_ratios= []
            iXGd_x_exp_ratios= []
            dLif_x_exp_ratios= []
            lwrp_x_exp_ratios= []
            smoo_x_exp_ratios= []
            vnGd_x_exp_ratios= []
            gdBp_x_exp_ratios= []
            occl_x_exp_ratios= []
            

        # ------------------------------------ distance of each explanation from the mean of explanations 
        for j in range(runs):
            t_x_exp_ratios.append(lp_norm_dif(t_x_exps_mean, t_x_exps[j], 
                                                 p_norm=descriptor['p_norm'], norm=False))
            
            shap_x_exp_ratios.append(lp_norm_dif(shap_x_exps_mean, shap_x_exps[j], 
                                                 p_norm=descriptor['p_norm'], norm=False))
            
            shap_krnel_x_exp_ratios.append(lp_norm_dif(shap_krnel_x_exps_mean, shap_krnel_x_exps[j], 
                                                 p_norm=descriptor['p_norm'], norm=False))
            
            if (data.shape[1]< 16):
                shap_exact_x_exp_ratios.append(lp_norm_dif(shap_exact_x_exps_mean, shap_exact_x_exps[j], 
                                                 p_norm=descriptor['p_norm'], norm=False))

            lime_x_exp_ratios.append(lp_norm_dif(lime_x_exps_mean, lime_x_exps[j], 
                                                 p_norm=descriptor['p_norm'], norm=False))
            
            if (is_model_NN==True):
                itGd_x_exp_ratios.append(lp_norm_dif(itGd_x_exps_mean, itGd_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
                
                iXGd_x_exp_ratios.append(lp_norm_dif(iXGd_x_exps_mean, iXGd_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
                
                dLif_x_exp_ratios.append(lp_norm_dif(dLif_x_exps_mean, dLif_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
                
                lwrp_x_exp_ratios.append(lp_norm_dif(lwrp_x_exps_mean, lwrp_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
                
                smoo_x_exp_ratios.append(lp_norm_dif(smoo_x_exps_mean, smoo_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
                
                vnGd_x_exp_ratios.append(lp_norm_dif(vnGd_x_exps_mean, vnGd_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
                
                gdBp_x_exp_ratios.append(lp_norm_dif(gdBp_x_exps_mean, gdBp_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
                
                occl_x_exp_ratios.append(lp_norm_dif(occl_x_exps_mean, occl_x_exps[j], 
                                              p_norm=descriptor['p_norm'], norm=False))
        
            
        # ------------------------------------ max ratio related to each x_data
        t_stability_ratios.append(t_x_exp_ratios[np.argmax(t_x_exp_ratios)])
        shap_stability_ratios.append(shap_x_exp_ratios[np.argmax(shap_x_exp_ratios)])
        shap_krnel_stability_ratios.append(shap_krnel_x_exp_ratios[np.argmax(shap_krnel_x_exp_ratios)])
        lime_stability_ratios.append(lime_x_exp_ratios[np.argmax(lime_x_exp_ratios)])
        
        if (data.shape[1]< 16):
            shap_exact_stability_ratios.append(shap_exact_x_exp_ratios[np.argmax(shap_exact_x_exp_ratios)])
        
        if (is_model_NN==True):
            itGd_stability_ratios.append(itGd_x_exp_ratios[np.argmax(itGd_x_exp_ratios)])
            iXGd_stability_ratios.append(iXGd_x_exp_ratios[np.argmax(iXGd_x_exp_ratios)])
            dLif_stability_ratios.append(dLif_x_exp_ratios[np.argmax(dLif_x_exp_ratios)])
            lwrp_stability_ratios.append(lwrp_x_exp_ratios[np.argmax(lwrp_x_exp_ratios)])
            smoo_stability_ratios.append(smoo_x_exp_ratios[np.argmax(smoo_x_exp_ratios)])
            vnGd_stability_ratios.append(vnGd_x_exp_ratios[np.argmax(vnGd_x_exp_ratios)])
            gdBp_stability_ratios.append(gdBp_x_exp_ratios[np.argmax(gdBp_x_exp_ratios)])
            occl_stability_ratios.append(occl_x_exp_ratios[np.argmax(occl_x_exp_ratios)])
        
        """
        print('instance number', i_data)
        print('shap_x_exps\n', shap_x_exps)
        print('lime_x_exps\n', lime_x_exps)
        print('shap_x_exps_mean\n', shap_x_exps_mean)
        print('lime_x_exps_mean\n', lime_x_exps_mean)
        print('shap_x_exp_ratios\n', shap_x_exp_ratios)
        print('lime_x_exp_ratios\n', lime_x_exp_ratios)
        print('shap_stab_ratios\n', shap_stability_ratios)
        print('lime_stab_ratios\n', lime_stability_ratios)
        """
    
    # ------------------------------------ general max ratio related to all data
    t_max = t_stability_ratios[np.argmax(t_stability_ratios)]
    shap_max= shap_stability_ratios[np.argmax(shap_stability_ratios)]
    shap_krnel_max= shap_krnel_stability_ratios[np.argmax(shap_krnel_stability_ratios)]
    lime_max= lime_stability_ratios[np.argmax(lime_stability_ratios)]
    
    if (data.shape[1]< 16):
        shap_exact_max= shap_exact_stability_ratios[np.argmax(shap_exact_stability_ratios)]
    else:
        shap_exact_max= '--'
        
    results= {
        'texp_res': t_max,
        'shap_res': shap_max,
        'shap_kernel_res': shap_krnel_max,
        'shap_exact_res': shap_exact_max,
        'lime_res': lime_max
    }
    
    if (is_model_NN==True):
        itGd_max= itGd_stability_ratios[np.argmax(itGd_stability_ratios)]
        iXGd_max= iXGd_stability_ratios[np.argmax(iXGd_stability_ratios)]
        dLif_max= dLif_stability_ratios[np.argmax(dLif_stability_ratios)]
        lwrp_max= lwrp_stability_ratios[np.argmax(lwrp_stability_ratios)]
        smoo_max= smoo_stability_ratios[np.argmax(smoo_stability_ratios)]
        vnGd_max= vnGd_stability_ratios[np.argmax(vnGd_stability_ratios)]
        gdBp_max= gdBp_stability_ratios[np.argmax(gdBp_stability_ratios)]
        occl_max= occl_stability_ratios[np.argmax(occl_stability_ratios)]
        
        results_grad= {
            'itGd_res': itGd_max,
            'iXGd_res': iXGd_max,
            'dLif_res': dLif_max,
            'lwrp_res': lwrp_max,
            'smoothG_res': smoo_max,
            'vanillaG_res': vnGd_max,
            'GuidBprop_res': gdBp_max,
            'occlusion_res': occl_max
        }
        
        results.update(results_grad)

    # max stability_ratios
    return results

In [67]:
# evaluate explanation stability from multiple runs over non-perturbed data -- using 2 instances from train_ox
print('RES --')
run_stability(nn3_model_ox, train_ox, labels_train_ox, descriptor_ox, is_model_NN=True)

RES --


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


...

  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


  0%|          | 0/1 [00:00<?, ?it/s]

Setting backward hooks on ReLU activations.The hooks will be removed after the attribution is finished


{'texp_res': 2.0187769454097274e-15,
 'shap_res': 0.12800373909469293,
 'shap_kernel_res': 0.035046489197427186,
 'shap_exact_res': '--',
 'lime_res': 0.02374042031916587,
 'itGd_res': 4.478635751439239e-15,
 'iXGd_res': 4.780742e-06,
 'dLif_res': 4.5115835e-06,
 'lwrp_res': 5.349247e-06,
 'smoothG_res': 0.5063961827813315,
 'vanillaG_res': 9.183178e-06,
 'GuidBprop_res': 9.183178e-06,
 'occlusion_res': 2.1567114e-06}

In [21]:
synth_ox= pd.read_csv('data/synth_OX_20.csv')

# split synth into features (x) and target (y)
df_inputs= synth_ox.loc[:,synth_ox.columns[0:20]]
df_labels= synth_ox.loc[:,synth_ox.columns[20:21]]

# split df_inputs and df_labels into train (80%) and test (20%) datasets
train_ox, test_ox, labels_train_ox, labels_test_ox= sklearn.model_selection.train_test_split(df_inputs,
                                                                                             df_labels,
                                                                                             train_size=0.80,
                                                                                             random_state=1234)
train_ox.head()

,ft_1,ft_2,ft_3,ft_4,ft_5,ft_6,ft_7,ft_8,ft_9,ft_10,ft_11,ft_12,ft_13,ft_14,ft_15,ft_16,ft_17,ft_18,ft_19,ft_20
281,0.705423,0.250877,0.393413,0.353529,0.328286,0.374804,0.312217,0.130608,0.281738,0.368790,0.758551,0.608015,0.440911,0.481930,0.263193,0.348485,0.127619,0.420548,0.382225,0.347964
42,0.119699,0.665472,0.235255,0.404961,0.292727,0.321720,0.246230,0.324422,0.360535,0.228708,0.630680,0.377279,0.328190,0.445910,0.344165,0.403461,0.579152,0.558943,0.448307,0.455760
255,0.226558,0.670225,0.344595,0.497451,0.298231,0.316184,0.430817,0.313977,0.414127,0.390934,0.637601,0.566323,0.420372,0.513464,0.609983,0.332080,0.327387,0.239984,0.463658,0.595457
906,0.229718,0.291506,0.394624,0.390758,0.310606,0.680242,0.283559,0.345412,0.421726,0.394265,0.589889,0.494661,0.640523,0.478399,0.437791,0.589921,0.648241,0.700112,0.269150,0.385109
394,0.373117,0.249822,0.259671,0.329655,0.307589,0.657956,0.408162,0.318289,0.333588,0.428981,0.631567,0.439209,0.392151,0.282901,0.562405,0.525431,0.416705,0.713325,0.586582,0.639019


In [22]:
from sklearn.neural_network import MLPClassifier

# Create the 3-hyden layers Neural Net classifier model based on OpenXAI synthetic
nn3_model_ox= MLPClassifier(activation='relu', alpha=0.0001, hidden_layer_sizes=(64, 64, 64), 
                            learning_rate_init=0.01, max_iter=500, random_state=0, solver='sgd')

nn3_model_ox.fit(train_ox, labels_train_ox.values.ravel())

acc_nn3_ox= sklearn.metrics.accuracy_score(labels_test_ox.values.ravel(), nn3_model_ox.predict(test_ox))
acc_nn3_ox

0.835

In [24]:
# get n and m parameters from train and labels_train
n_ox, m_ox= texp.get_n_m_sizes(train_ox, labels_train_ox)

# conversion of train_ox and labels_train_ox data back to tensor as required by benchmarking methods
# explanations will be only done over train dataset
tn_train= torch.from_numpy(train_ox.values)
tn_lb_tr= torch.from_numpy(labels_train_ox.values.ravel().astype(int))

In [25]:
# define a descriptor to OpenXAI synthetic

descriptor_ox= dict()

h_min_dist_ox= texp.get_minimum_distance(train_ox)

# T-Exp explanation settings
descriptor_ox['h_min']= h_min_dist_ox
descriptor_ox['h_max']= 1
descriptor_ox['jacobian_eps']= 1e-3
descriptor_ox['max_itr']= 30
descriptor_ox['ohe_delta']= 0

# perturbation settings used for the stability metric
descriptor_ox['num_samples']= 50
descriptor_ox['num_perts']= 20
descriptor_ox['pert_max_distance']= (h_min_dist_ox/2)
descriptor_ox['num_runs']= 25
descriptor_ox['feature_metadata']= ['c'] * n_ox
descriptor_ox['p_norm']= 2
descriptor_ox['eps_norm']= 1e-6
descriptor_ox['top_k']= 0
descriptor_ox['mask']= generate_mask(tn_train[0].reshape(-1), descriptor_ox['top_k'])

In [33]:
exp= explainer.Explainers()

# SHAP Explainer from SHAP

target_i= pd.DataFrame(data=[train_ox.iloc[0,:]], columns=train_ox.columns)

shap_values= exp.shap(nn3_model_ox, train_ox, target_i)
print("shap_values =", shap_values)

shap_values = tensor([-0.1604, -0.0458, -0.0546, -0.1204,  0.0096,  0.0054,  0.0017, -0.0738,
        -0.0442, -0.0571,  0.0963, -0.0025, -0.0075,  0.0063, -0.0617,  0.0046,
         0.1271,  0.0046, -0.1029, -0.0246], dtype=torch.float64)


In [34]:
# KernelSHAP Explainer from SHAP

shap_values= exp.k_shap(nn3_model_ox, train_ox, target_i)
print("shap_values =", shap_values)

  0%|          | 0/1 [00:00<?, ?it/s]

shap_values = tensor([-0.1572, -0.0533, -0.0727, -0.1105,  0.0074,  0.0124, -0.0048, -0.0973,
        -0.0228, -0.0377,  0.0987, -0.0018, -0.0047,  0.0077, -0.0722,  0.0000,
         0.1227,  0.0109, -0.1048, -0.0199], dtype=torch.float64)


In [35]:
# KernelSHAP from captum

nn_pytorch_model_ox= sklearn_to_pytorch_NN(nn3_model_ox, train_ox.shape[1])

target_i= pd.DataFrame(data=[train_ox.iloc[0,:]], columns=train_ox.columns)
x_data_tensor= torch.tensor(np.asarray(target_i), dtype=torch.float32)

# Explain predictions on test data
attributions= exp.c_kshap(nn_pytorch_model_ox, x_data_tensor)

# Print attributions for a sample data point
print("SHAP Attributions:", attributions)

SHAP Attributions: tensor([ 1.6970,  0.4748, -4.2497,  2.5004, -0.7886,  0.5051,  2.2549,  1.4237,
        -0.6858, -5.1411,  0.6074, -1.3344,  0.5587, -0.4708, -1.4206, -0.8322,
        -0.3052, -1.2382,  1.7670, -0.1643])
